# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/content-refresh-prioritization/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Content Refresh Prioritization. Framed as an ML task, this is a classification problem: for each page, predict whether it belongs in the "needs review" class or not. It could also be framed as a ranking/scoring problem — producing a priority score per page so the content team can sort and work down a queue — which is actually closer to how the real pipeline's refresh_queue output works. I'll treat it primarily as a scoring/ranking task, since the deliverable is an ordered list of pages to review, not just a yes/no label.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no direct "should be refreshed" label in the data, so I need a proxy target. A reasonable proxy: a page is "stale" if it has high days_since_last_update AND declining or flat performance (e.g. low or dropping ctr, or a negative trend_direction). This mirrors the label the starter pipeline's baseline rule already approximates, and is a stand-in for what a human reviewer would actually flag by hand.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'll use Precision@50, matching what the starter pipeline already computes: out of the top 50 pages the model ranks as highest priority, what fraction are genuinely worth reviewing? This metric fits the real-world action directly, since a content team only has time to review a limited number of pages per cycle, so precision at the top of the queue matters more than overall accuracy across all 30,000 pages.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/faith-amanze/content-refresh-prioritization/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Unit of analysis: one row = one page")
print(f"Shape: {df.shape}")
df.head()

Unit of analysis: one row = one page
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

This 167-day-AND-below-median-CTR rule, run below, only flags 120 of 30,000 pages
(0.4%) as `proxy_needs_review`. That's the tell: an if-statement needs a hard cutoff on
every condition, and both cutoffs here are arbitrary -- there's nothing principled about
167 days specifically, or about requiring the CTR condition to ALSO be true rather than
letting a page qualify through staleness alone with a strong-enough signal elsewhere. A
page at 160 days with terrible CTR gets skipped just as easily as one at 104 days sitting
right at the CTR median (row 9 in the table below is a near-miss like this). A model
doesn't need one hard-and-fast rule -- it can weigh many signals (staleness, CTR, position,
volume, content_type) together and let a page with a strong signal on one dimension still
rank high even without clearing every fixed threshold. That's the real argument for ML
here: not that the rule is wrong, but that reality doesn't split cleanly along the two lines
this if-statement draws.

In [2]:
# Sketch a proxy target column
df['proxy_needs_review'] = (
    (df['days_since_last_update'] > 167) &
    (df['ctr'] < df['ctr'].median())
).astype(int)

print(df['proxy_needs_review'].value_counts())
df[['days_since_last_update', 'ctr', 'proxy_needs_review']].head(10)

proxy_needs_review
0    29880
1      120
Name: count, dtype: int64


,days_since_last_update,ctr,proxy_needs_review
0,20,0.76,0
1,25,0.05,0
2,20,0.09,0
3,22,0.49,0
4,14,0.13,0
5,20,0.03,0
6,20,0.00,0
7,22,0.06,0
8,20,0.09,0
9,104,0.16,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.